# Notebook 3: DCM Bottom-Layer Model

**Time:** 60--90 minutes

**Purpose:** Replace the current mean-then-Bernoulli observation layer with an ordinal probit model. Build a proof-of-concept PyMC implementation on a small tree and verify parameter recovery.

**Prerequisites:** Notebooks 1--2 (latent-threshold models, identifiability, SDT mapping, leaf-node likelihood).

**What changes:** The current DCM takes an expert-averaged probability, draws a single Bernoulli, and repeats via outer Monte Carlo. The new layer takes raw ordinal ratings from each expert, models them jointly with shared cutpoints, and marginalises the latent indicator state inside the model.

In [ ]:
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.special import logsumexp

rng = np.random.default_rng(42)


def ordinal_probs(cutpoints: np.ndarray, eta: float = 0.0,
                  cdf: str = "probit") -> np.ndarray:
    """Compute ordinal category probabilities from cutpoints and linear predictor."""
    c_aug = np.concatenate([[-np.inf], cutpoints, [np.inf]])
    if cdf == "probit":
        return np.diff(norm.cdf(c_aug - eta))
    elif cdf == "logistic":
        from scipy.special import expit
        return np.diff(expit(c_aug - eta))
    raise ValueError(f"Unknown cdf: {cdf}")

### Learning objectives

1. Identify the limitations of the current mean-then-Bernoulli observation layer.
2. Fit an ordinal probit model with known and unknown latent states in PyMC.
3. Implement the latent-state marginalisation using `pm.Potential` and log-sum-exp.
4. Build a proof-of-concept mini DCM tree with the new observation model.
5. Verify parameter recovery on synthetic data.

---

## Part A: What's wrong with the current approach?

The current `dcm_model.py` observation layer works as follows:

1. For each indicator, **average** all expert probability judgements into a single number.
2. Draw a **single Bernoulli** from that average.
3. Repeat everything via **outer Monte Carlo** to propagate uncertainty.

Three problems:

| Issue | Consequence |
|---|---|
| **Averaging** collapses expert responses | Loses inter-expert disagreement; 3 experts saying 0.5 looks the same as one saying 0.1 and one saying 0.9 |
| **Single Bernoulli** per indicator | Massive information loss -- a continuous probability is reduced to one bit |
| **Outer Monte Carlo** | Computationally expensive; requires many full MCMC runs to propagate observation-level uncertainty |

The ordinal probit model fixes all three: each expert's raw rating enters the likelihood directly, expert heterogeneity is modelled via bias parameters, and the latent indicator state is marginalised inside a single MCMC run.

---

## Part B: Ordinal probit with known latent state

We start with the simplest case: the latent indicator state $z$ is **known** (observed as a covariate). This is the model from Notebook 2, Exercise 3. We repeat it here as the foundation for the unknown-$z$ case.

### Model specification

For expert $e$ rating indicator $j$ on system $s$:

$$u_{esj} \mid z_{sj} \sim \mathcal{N}(b_e + a \cdot z_{sj}, \; 1)$$

$$P(r_{esj} = k \mid z_{sj}) = \Phi(\kappa_k - b_e - a \cdot z_{sj}) - \Phi(\kappa_{k-1} - b_e - a \cdot z_{sj})$$

**Identifiability constraints** (from Notebook 2):
- Variance fixed to 1 (scale anchor).
- $b_1 = 0$ (location anchor for expert biases).
- Cutpoints $\kappa$ are ordered.

In [ ]:
# --- Synthetic data: known z ---
n_experts: int = 3
n_indicators: int = 12
K: int = 7  # 7-point scale

# True parameters
true_a: float = 1.5         # discrimination
true_b: np.ndarray = np.array([0.0, 0.5, -0.3])  # expert biases (b_1 = 0 anchor)
true_kappa: np.ndarray = np.array([-1.5, -0.8, -0.1, 0.4, 1.0, 1.6])  # 6 cutpoints

# Latent indicator states (known)
true_z: np.ndarray = rng.binomial(1, 0.5, size=n_indicators)

# Generate ratings
expert_idx_list: list[int] = []
indicator_idx_list: list[int] = []
ratings_list: list[int] = []

for e in range(n_experts):
    for j in range(n_indicators):
        eta_ej: float = true_b[e] + true_a * true_z[j]
        p_ej = ordinal_probs(true_kappa, eta=eta_ej)
        r = rng.choice(K, p=p_ej)
        expert_idx_list.append(e)
        indicator_idx_list.append(j)
        ratings_list.append(r)

expert_idx: np.ndarray = np.array(expert_idx_list)
indicator_idx: np.ndarray = np.array(indicator_idx_list)
ratings: np.ndarray = np.array(ratings_list)

print(f"Observations: {len(ratings)}")
print(f"True z: {true_z}")
print(f"Rating distribution: {np.bincount(ratings, minlength=K)}")

In [ ]:
# PyMC model: known z
with pm.Model() as known_z_model:
    # Discrimination (positive)
    a = pm.HalfNormal("a", sigma=3.0)

    # Expert biases: b_1 = 0 (anchor), rest free
    b_free = pm.Normal("b_free", mu=0.0, sigma=2.0, shape=n_experts - 1)
    b = pt.concatenate([pt.zeros(1), b_free])

    # Shared ordered cutpoints
    kappa = pm.Normal(
        "kappa", mu=0.0, sigma=2.0, shape=K - 1,
        transform=pm.distributions.transforms.ordered,
        initval=np.linspace(-1.5, 1.5, K - 1),
    )

    # Linear predictor: z is known
    eta = b[expert_idx] + a * true_z[indicator_idx]

    pm.OrderedProbit("r", eta=eta, cutpoints=kappa, observed=ratings)

with known_z_model:
    idata_known = pm.sample(draws=1000, tune=1000, chains=4, random_seed=42)

In [ ]:
# Parameter recovery
print("=== Discrimination ===")
a_post = idata_known.posterior["a"].mean().item()
print(f"  True: {true_a:.2f}    Posterior: {a_post:.3f}")

print("\n=== Expert biases (free) ===")
b_post = idata_known.posterior["b_free"].mean(dim=("chain", "draw")).values
print(f"  True: {true_b[1:]}    Posterior: {np.round(b_post, 3)}")

print("\n=== Cutpoints ===")
k_post = idata_known.posterior["kappa"].mean(dim=("chain", "draw")).values
print(f"  True:  {true_kappa}")
print(f"  Post:  {np.round(k_post, 3)}")

az.summary(idata_known, var_names=["a", "b_free", "kappa"])

---

## Part C: Unknown $z$ -- marginalising the latent state

In the real DCM, the latent indicator state $z_{sj}$ is **not observed**. The tree above provides a prior probability $q_{sj} = P(z_{sj} = 1)$. We must marginalise $z$ out of the likelihood.

From Notebook 2:

$$P(\mathbf{r}_{sj} \mid q, \theta) = (1 - q)\prod_e P(r_{esj} \mid z=0, \theta) \;+\; q \prod_e P(r_{esj} \mid z=1, \theta)$$

### Implementation strategy

PyMC cannot directly sample discrete latent variables in its default NUTS sampler. Instead, we **marginalise** $z$ analytically and add the log-marginal-likelihood as a `pm.Potential`. This is the standard approach for discrete latent variables in HMC-based samplers.

For each indicator $j$, the contribution to the log-likelihood is:

$$\log P(\mathbf{r}_j \mid q_j, \theta) = \text{logsumexp}\!\Big(\log(1-q_j) + \sum_e \log P(r_{ej} \mid z=0), \;\; \log q_j + \sum_e \log P(r_{ej} \mid z=1)\Big)$$

In [ ]:
# Use the SAME synthetic data as Part B, but now treat z as unknown.
# We give the model q_j = P(z_j = 1) as a prior -- here, use a learnable q per indicator.

# First, a helper to compute ordered-probit log-probs in pytensor.
def pt_ordinal_logp(rating: pt.TensorVariable,
                    kappa: pt.TensorVariable,
                    eta: pt.TensorVariable) -> pt.TensorVariable:
    """Log P(rating = k | eta, kappa) for ordered probit, in pytensor.

    Parameters
    ----------
    rating : int tensor, shape (N,), 0-indexed category.
    kappa  : tensor, shape (K-1,), ordered cutpoints.
    eta    : tensor, shape (N,), linear predictor.

    Returns
    -------
    Log-probability tensor, shape (N,).
    """
    # Augment cutpoints with +/- large values
    kappa_aug = pt.concatenate([pt.full((1,), -20.0), kappa, pt.full((1,), 20.0)])
    # CDF values at augmented cutpoints
    # Phi(kappa_k - eta) for each observation
    upper = pt.erfc(-(kappa_aug[rating + 1] - eta) / pt.sqrt(2.0)) / 2.0
    lower = pt.erfc(-(kappa_aug[rating] - eta) / pt.sqrt(2.0)) / 2.0
    prob = upper - lower
    return pt.log(pt.clip(prob, 1e-12, 1.0))

In [ ]:
# Restructure data for marginalisation: group ratings by indicator
# ratings_by_indicator[j] = list of (expert_idx, rating) pairs
ratings_by_indicator: dict[int, list[tuple[int, int]]] = {}
for i in range(len(ratings)):
    j = indicator_idx[i]
    if j not in ratings_by_indicator:
        ratings_by_indicator[j] = []
    ratings_by_indicator[j].append((expert_idx[i], ratings[i]))

# Build flat arrays indexed by indicator for the marginalisation
# For each indicator j, we need all expert indices and ratings
ind_expert_idx: list[np.ndarray] = []
ind_ratings: list[np.ndarray] = []
for j in range(n_indicators):
    pairs = ratings_by_indicator[j]
    ind_expert_idx.append(np.array([p[0] for p in pairs]))
    ind_ratings.append(np.array([p[1] for p in pairs]))

print(f"Indicator 0: experts={ind_expert_idx[0]}, ratings={ind_ratings[0]}")
print(f"True z[0]={true_z[0]}")

In [ ]:
# PyMC model: unknown z, marginalised via pm.Potential
with pm.Model() as marginal_model:
    # --- Observation model parameters ---
    a = pm.HalfNormal("a", sigma=3.0)
    b_free = pm.Normal("b_free", mu=0.0, sigma=2.0, shape=n_experts - 1)
    b = pt.concatenate([pt.zeros(1), b_free])
    kappa = pm.Normal(
        "kappa", mu=0.0, sigma=2.0, shape=K - 1,
        transform=pm.distributions.transforms.ordered,
        initval=np.linspace(-1.5, 1.5, K - 1),
    )

    # --- Per-indicator prior on z ---
    # In the full DCM, q comes from the tree. Here we learn it per indicator.
    q = pm.Beta("q", alpha=1.0, beta=1.0, shape=n_indicators)

    # --- Marginalised log-likelihood ---
    log_lik_terms = []
    for j in range(n_indicators):
        e_idx_j = ind_expert_idx[j]
        r_j = ind_ratings[j]

        # Log-likelihood under z=0 and z=1
        for z_val in [0, 1]:
            eta_j = b[e_idx_j] + a * z_val
            logp_z = pt_ordinal_logp(
                pt.as_tensor_variable(r_j),
                kappa,
                eta_j,
            )
            if z_val == 0:
                ll_z0 = pt.log(1.0 - q[j]) + pt.sum(logp_z)
            else:
                ll_z1 = pt.log(q[j]) + pt.sum(logp_z)

        # logsumexp over z
        log_lik_j = pt.logaddexp(ll_z0, ll_z1)
        log_lik_terms.append(log_lik_j)

    total_ll = pt.sum(pt.stack(log_lik_terms))
    pm.Potential("marginal_ll", total_ll)

with marginal_model:
    idata_marginal = pm.sample(draws=1000, tune=1500, chains=4,
                                random_seed=42, target_accept=0.9)

In [ ]:
# Compare observation model parameters to known-z fit
print("=== Discrimination (a) ===")
a_marg = idata_marginal.posterior["a"].mean().item()
print(f"  True: {true_a:.2f}    Known-z: {a_post:.3f}    Marginal: {a_marg:.3f}")

print("\n=== Expert biases (b_free) ===")
b_marg = idata_marginal.posterior["b_free"].mean(dim=("chain", "draw")).values
print(f"  True: {true_b[1:]}")
print(f"  Known-z:  {np.round(b_post, 3)}")
print(f"  Marginal: {np.round(b_marg, 3)}")

print("\n=== Cutpoints ===")
k_marg = idata_marginal.posterior["kappa"].mean(dim=("chain", "draw")).values
print(f"  True:     {true_kappa}")
print(f"  Known-z:  {np.round(k_post, 3)}")
print(f"  Marginal: {np.round(k_marg, 3)}")

In [ ]:
# Posterior q vs true z: does the model recover the latent states?
q_post = idata_marginal.posterior["q"].mean(dim=("chain", "draw")).values
q_hdi = az.hdi(idata_marginal, var_names=["q"])["q"].values

fig, ax = plt.subplots(figsize=(10, 4))
for j in range(n_indicators):
    colour = "C0" if true_z[j] == 0 else "C3"
    ax.errorbar(j, q_post[j],
                yerr=[[q_post[j] - q_hdi[j, 0]], [q_hdi[j, 1] - q_post[j]]],
                fmt="o", color=colour, capsize=4)

ax.axhline(0.5, color="grey", ls="--", alpha=0.3)
ax.set_xticks(range(n_indicators))
ax.set_xlabel("Indicator $j$")
ax.set_ylabel("Posterior $q_j = P(z_j = 1)$")
ax.set_title("Latent state recovery: blue = true $z=0$, red = true $z=1$")
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

**Key observations:**

- The observation model parameters ($a$, $b$, $\kappa$) are recovered well even when $z$ is unknown -- the marginalisation works.
- The posterior $q_j$ separates the two latent states: indicators with $z_j = 1$ (red) should have high $q_j$, and $z_j = 0$ (blue) should have low $q_j$.
- With only 3 experts per indicator, some uncertainty remains. More experts or more extreme $a$ would sharpen the separation.

---

## Part D: Mini DCM tree proof-of-concept

We now embed the ordinal probit observation model in a small DCM-like tree to demonstrate end-to-end inference.

### Tree structure

```
Stance (p_stance ~ Beta)
├── Feature 1 (z_1 ~ Bernoulli | stance)
│   ├── Indicator 1a  ← expert ratings
│   └── Indicator 1b  ← expert ratings
└── Feature 2 (z_2 ~ Bernoulli | stance)
    ├── Indicator 2a  ← expert ratings
    └── Indicator 2b  ← expert ratings
```

- **Stance** has a Beta prior on its probability.
- Each **feature** has a latent binary state. When the stance is present, the feature is more likely present.
- Each **indicator** has a latent binary state driven by its parent feature. Expert ratings at the leaves are generated by the ordinal probit model.

This is a simplified version of the full DCM, but it demonstrates the key mechanism: expert ratings at the leaves propagate upward to inform the stance probability.

In [ ]:
# --- Synthetic data for mini DCM tree ---
# True parameters
true_p_stance: float = 0.7
true_stance: int = 1  # stance is present

# Feature presence probabilities conditional on stance
# P(feature | stance=1) is high, P(feature | stance=0) is low
p_feature_given_stance = {1: 0.85, 0: 0.15}

# Indicator presence conditional on feature
p_indicator_given_feature = {1: 0.9, 0: 0.1}

# Observation model parameters (same as before)
tree_a: float = 1.5
tree_b: np.ndarray = np.array([0.0, 0.5, -0.3])
tree_kappa: np.ndarray = np.array([-1.5, -0.8, -0.1, 0.4, 1.0, 1.6])
n_tree_experts: int = 3

# Generate the tree top-down
true_features: np.ndarray = np.array([
    rng.binomial(1, p_feature_given_stance[true_stance])
    for _ in range(2)
])

# 2 indicators per feature = 4 indicators total
true_indicators: np.ndarray = np.array([
    rng.binomial(1, p_indicator_given_feature[true_features[f]])
    for f in range(2)
    for _ in range(2)
])

# Generate expert ratings for each indicator
tree_expert_idx: list[int] = []
tree_indicator_idx: list[int] = []
tree_ratings: list[int] = []

for ind_j in range(4):
    for e in range(n_tree_experts):
        eta = tree_b[e] + tree_a * true_indicators[ind_j]
        p = ordinal_probs(tree_kappa, eta=eta)
        r = rng.choice(K, p=p)
        tree_expert_idx.append(e)
        tree_indicator_idx.append(ind_j)
        tree_ratings.append(r)

tree_expert_idx = np.array(tree_expert_idx)
tree_indicator_idx = np.array(tree_indicator_idx)
tree_ratings = np.array(tree_ratings)

print(f"True stance: {true_stance}")
print(f"True features: {true_features}")
print(f"True indicators: {true_indicators}")
print(f"Ratings: {tree_ratings}")
print(f"Total observations: {len(tree_ratings)}")

### Full marginalisation strategy

Every discrete latent variable in the tree must be marginalised out. The tree has:
- 1 stance (continuous $p_{\text{stance}}$ -- sampled by NUTS)
- 2 features ($z_f \in \{0,1\}$ -- marginalised)
- 4 indicators ($z_i \in \{0,1\}$ -- marginalised)

We marginalise the indicators inside each feature, then marginalise the features. The structure factorises because indicators are conditionally independent given their parent feature.

For feature $f$ with indicators $\{i_1, i_2\}$:

$$P(\text{ratings}_f \mid z_f, \theta) = \prod_{i \in \{i_1, i_2\}} P(\text{ratings}_i \mid z_f, \theta)$$

where each indicator's contribution is itself a marginalisation over $z_i$:

$$P(\text{ratings}_i \mid z_f, \theta) = \sum_{z_i} P(z_i \mid z_f) \prod_e P(r_{ei} \mid z_i, \theta)$$

Then for each feature:

$$P(\text{ratings}_f \mid p_s, \theta) = \sum_{z_f} P(z_f \mid p_s) \; P(\text{ratings}_f \mid z_f, \theta)$$

All sums are over $\{0, 1\}$, so the total computation is cheap.

In [ ]:
# Group ratings by indicator for the tree model
tree_ind_ratings: list[np.ndarray] = []
tree_ind_experts: list[np.ndarray] = []
for j in range(4):
    mask = tree_indicator_idx == j
    tree_ind_ratings.append(tree_ratings[mask])
    tree_ind_experts.append(tree_expert_idx[mask])

# Tree structure: which indicators belong to which feature
feature_to_indicators: list[list[int]] = [[0, 1], [2, 3]]

# Conditional probabilities (as log-probs for the pytensor computation)
# P(z_feature | stance), P(z_indicator | z_feature)
p_feat_given_s = {1: 0.85, 0: 0.15}  # known/fixed for this proof-of-concept
p_ind_given_f = {1: 0.9, 0: 0.1}

print("Tree structure:")
for f_idx, inds in enumerate(feature_to_indicators):
    print(f"  Feature {f_idx} -> Indicators {inds}")
    for i in inds:
        print(f"    Indicator {i}: ratings = {tree_ind_ratings[i]}")

In [ ]:
# Full mini-DCM model with ordinal probit leaves
with pm.Model() as mini_dcm:
    # --- Stance prior ---
    p_stance = pm.Beta("p_stance", alpha=1.0, beta=1.0)

    # --- Observation model parameters ---
    a = pm.HalfNormal("a", sigma=3.0)
    b_free = pm.Normal("b_free", mu=0.0, sigma=2.0, shape=n_tree_experts - 1)
    b = pt.concatenate([pt.zeros(1), b_free])
    kappa = pm.Normal(
        "kappa", mu=0.0, sigma=2.0, shape=K - 1,
        transform=pm.distributions.transforms.ordered,
        initval=np.linspace(-1.5, 1.5, K - 1),
    )

    # --- Marginalised tree log-likelihood ---
    # For each stance value s in {0, 1}, compute total log-lik of all ratings
    log_lik_stance = []  # will hold log P(all ratings | stance=s, theta) for s=0,1

    for s_val in [0, 1]:
        log_p_s = pt.switch(pt.eq(s_val, 1), pt.log(p_stance), pt.log(1.0 - p_stance))

        # Accumulate log-lik across features
        log_lik_features = pt.zeros(())
        for f_idx, ind_list in enumerate(feature_to_indicators):

            # For each feature value z_f in {0, 1}
            log_lik_zf = []
            for z_f in [0, 1]:
                log_p_zf = np.log(p_feat_given_s[z_f]) if s_val == z_f \
                    else np.log(1 - p_feat_given_s[1 - z_f])
                # P(z_f | stance=s_val)
                if s_val == 1:
                    log_p_zf = np.log(p_feat_given_s[z_f])  # P(z_f=z_f | s=1)
                else:
                    log_p_zf = np.log(p_feat_given_s[z_f])  # same structure, different meaning
                # Simpler: P(z_f=1|s) = p_feat_given_s[s], P(z_f=0|s) = 1-p_feat_given_s[s]
                if z_f == 1:
                    log_p_zf_val = np.log(p_feat_given_s[s_val])
                else:
                    log_p_zf_val = np.log(1.0 - p_feat_given_s[s_val])

                # For each indicator under this feature, marginalise z_indicator
                log_lik_inds = pt.zeros(())
                for ind_j in ind_list:
                    r_j = pt.as_tensor_variable(tree_ind_ratings[ind_j])
                    e_j = tree_ind_experts[ind_j]

                    # Marginalise z_indicator given z_feature
                    log_lik_zi = []
                    for z_i in [0, 1]:
                        if z_i == 1:
                            log_p_zi = np.log(p_ind_given_f[z_f])
                        else:
                            log_p_zi = np.log(1.0 - p_ind_given_f[z_f])

                        eta_i = b[e_j] + a * z_i
                        logp_r = pt_ordinal_logp(r_j, kappa, eta_i)
                        log_lik_zi.append(log_p_zi + pt.sum(logp_r))

                    log_lik_inds = log_lik_inds + pt.logaddexp(log_lik_zi[0], log_lik_zi[1])

                log_lik_zf.append(log_p_zf_val + log_lik_inds)

            log_lik_features = log_lik_features + pt.logaddexp(log_lik_zf[0], log_lik_zf[1])

        log_lik_stance.append(log_p_s + log_lik_features)

    total_ll = pt.logaddexp(log_lik_stance[0], log_lik_stance[1])
    pm.Potential("tree_ll", total_ll)

with mini_dcm:
    idata_tree = pm.sample(draws=1000, tune=1500, chains=4,
                            random_seed=42, target_accept=0.9)

In [ ]:
# Results: stance posterior
print("=== Stance probability ===")
p_stance_post = idata_tree.posterior["p_stance"].values.flatten()
print(f"  True stance: {true_stance} (generated with p_stance = {true_p_stance})")
print(f"  Posterior mean: {p_stance_post.mean():.3f}")
print(f"  Posterior sd:   {p_stance_post.std():.3f}")
print(f"  94% HDI: {az.hdi(idata_tree, var_names=['p_stance'])['p_stance'].values}")

# Observation model parameters
print("\n=== Observation model ===")
a_tree = idata_tree.posterior["a"].mean().item()
b_tree = idata_tree.posterior["b_free"].mean(dim=("chain", "draw")).values
k_tree = idata_tree.posterior["kappa"].mean(dim=("chain", "draw")).values
print(f"  a:  True={tree_a:.1f}   Post={a_tree:.3f}")
print(f"  b:  True={tree_b[1:]}   Post={np.round(b_tree, 3)}")
print(f"  kappa True: {tree_kappa}")
print(f"  kappa Post: {np.round(k_tree, 3)}")

In [ ]:
# Posterior density for p_stance
fig, ax = plt.subplots(figsize=(8, 3.5))
az.plot_posterior(idata_tree, var_names=["p_stance"], ax=ax, hdi_prob=0.94)
ax.axvline(true_p_stance, color="C3", ls="--", lw=2, label=f"True $p_{{stance}}$ = {true_p_stance}")
ax.legend()
ax.set_title("Posterior of stance probability")
plt.tight_layout()
plt.show()

**What just happened:** Expert ordinal ratings at the leaves propagated upward through two layers of discrete marginalisation to inform the stance probability at the root. No outer Monte Carlo was needed -- the entire inference is a single MCMC run.

---

## Exercises

### Exercise 1: What happens when $a = 0$? (pen-and-paper)

Suppose the discrimination parameter $a = 0$.

**(a)** Write out $P(r_{esj} = k \mid z = 0, \theta)$ and $P(r_{esj} = k \mid z = 1, \theta)$. What happens?

**(b)** What does this imply for the marginal likelihood $P(\mathbf{r} \mid q, \theta)$ as a function of $q$?

**(c)** What would you expect the posterior of $p_{\text{stance}}$ to look like if $a = 0$?

**Stop and try** before reading the solution.

---

<details>
<summary><b>Solution</b></summary>

**(a)** With $a = 0$: $\eta = b_e + 0 \cdot z = b_e$ regardless of $z$. The two components of the mixture produce identical rating probabilities: $P(r \mid z=0) = P(r \mid z=1)$.

**(b)** The marginal likelihood becomes:

$$P(\mathbf{r} \mid q, \theta) = (1-q) \prod_e P(r_e \mid b_e, \kappa) + q \prod_e P(r_e \mid b_e, \kappa) = \prod_e P(r_e \mid b_e, \kappa)$$

The $q$ terms cancel. The ratings carry **no information** about the latent state.

**(c)** The posterior of $p_{\text{stance}}$ would equal its prior (Beta(1,1) = Uniform). The data cannot discriminate between the stance being present or absent.

This is the SDT interpretation: $d' = 0$ means zero sensitivity -- the observer cannot distinguish signal from noise.

</details>

### Exercise 2: Sensitivity analysis -- more experts (code)

Re-run the marginal model from Part C with **6 experts** instead of 3 (add 3 more with biases of your choice). Compare:

1. How much tighter are the posteriors on $a$ and $\kappa$?
2. How much better is the $q_j$ separation between $z=0$ and $z=1$ indicators?

Use the same `true_a`, `true_kappa`, and `true_z` as before.

In [ ]:
# YOUR CODE HERE
# Suggested steps:
# 1. Define true_b_6 = np.array([0.0, 0.5, -0.3, 0.2, -0.6, 0.1])
# 2. Generate ratings for 6 experts x 12 indicators using true_z from Part B
# 3. Build the marginal model (same structure as Part C, but n_experts=6)
# 4. Compare posterior summaries and q_j plots


<details>
<summary><b>Solution</b></summary>

```python
# --- Data generation with 6 experts ---
n_experts_6 = 6
true_b_6 = np.array([0.0, 0.5, -0.3, 0.2, -0.6, 0.1])

expert_idx_6, indicator_idx_6, ratings_6 = [], [], []
for e in range(n_experts_6):
    for j in range(n_indicators):
        eta_ej = true_b_6[e] + true_a * true_z[j]
        p_ej = ordinal_probs(true_kappa, eta=eta_ej)
        r = rng.choice(K, p=p_ej)
        expert_idx_6.append(e)
        indicator_idx_6.append(j)
        ratings_6.append(r)
expert_idx_6 = np.array(expert_idx_6)
indicator_idx_6 = np.array(indicator_idx_6)
ratings_6 = np.array(ratings_6)

# Group by indicator
ind_expert_6, ind_ratings_6 = [], []
for j in range(n_indicators):
    mask = indicator_idx_6 == j
    ind_expert_6.append(expert_idx_6[mask])
    ind_ratings_6.append(ratings_6[mask])

# --- Model ---
with pm.Model() as marginal_6:
    a = pm.HalfNormal("a", sigma=3.0)
    b_free = pm.Normal("b_free", mu=0.0, sigma=2.0, shape=n_experts_6 - 1)
    b = pt.concatenate([pt.zeros(1), b_free])
    kappa = pm.Normal("kappa", mu=0.0, sigma=2.0, shape=K - 1,
                      transform=pm.distributions.transforms.ordered,
                      initval=np.linspace(-1.5, 1.5, K - 1))
    q = pm.Beta("q", alpha=1.0, beta=1.0, shape=n_indicators)

    ll_terms = []
    for j in range(n_indicators):
        for z_val in [0, 1]:
            eta_j = b[ind_expert_6[j]] + a * z_val
            logp_z = pt_ordinal_logp(pt.as_tensor_variable(ind_ratings_6[j]), kappa, eta_j)
            if z_val == 0:
                ll_z0 = pt.log(1.0 - q[j]) + pt.sum(logp_z)
            else:
                ll_z1 = pt.log(q[j]) + pt.sum(logp_z)
        ll_terms.append(pt.logaddexp(ll_z0, ll_z1))
    pm.Potential("marginal_ll", pt.sum(pt.stack(ll_terms)))

with marginal_6:
    idata_6 = pm.sample(draws=1000, tune=1500, chains=4, random_seed=42, target_accept=0.9)

# Compare
q6 = idata_6.posterior["q"].mean(dim=("chain", "draw")).values
q6_hdi = az.hdi(idata_6, var_names=["q"])["q"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, q_vals, q_h, title in [
    (axes[0], q_post, q_hdi, "3 experts"),
    (axes[1], q6, q6_hdi, "6 experts"),
]:
    for j in range(n_indicators):
        c = "C0" if true_z[j] == 0 else "C3"
        ax.errorbar(j, q_vals[j],
                    yerr=[[q_vals[j]-q_h[j,0]], [q_h[j,1]-q_vals[j]]],
                    fmt="o", color=c, capsize=4)
    ax.axhline(0.5, color="grey", ls="--", alpha=0.3)
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Indicator")
    ax.set_ylabel("$q_j$")
plt.tight_layout()
plt.show()

print("a posterior sd:  3 experts =", idata_marginal.posterior["a"].std().item():.3f,
      "  6 experts =", idata_6.posterior["a"].std().item():.3f)
```

</details>

### Exercise 3: Refactor the tree marginalisation (code)

The Part D tree model uses nested Python loops to enumerate all $z$ configurations. This works but does not scale well.

Write a function `tree_log_marginal_lik` that takes:
- `p_stance`: a pytensor scalar (the stance probability),
- `a`, `b`, `kappa`: observation model parameters (pytensor tensors),
- `feature_to_indicators`: the tree structure,
- `tree_ind_ratings`, `tree_ind_experts`: the data,
- `p_feat_given_s`, `p_ind_given_f`: conditional probabilities,

and returns the total log-marginal-likelihood as a single pytensor scalar. Then use it in a `pm.Potential` to reproduce the same results as Part D.

**Goal:** make the tree construction modular so it can be extended to larger trees.

In [ ]:
# YOUR CODE HERE


<details>
<summary><b>Solution</b></summary>

```python
def indicator_log_marginal(
    ratings: pt.TensorVariable,
    expert_idx: np.ndarray,
    z_parent: int,
    a: pt.TensorVariable,
    b: pt.TensorVariable,
    kappa: pt.TensorVariable,
    p_ind_given_f: dict[int, float],
) -> pt.TensorVariable:
    """Log P(ratings_i | z_parent, theta), marginalised over z_indicator."""
    ll_zi = []
    for z_i in [0, 1]:
        log_p_zi = np.log(p_ind_given_f[z_parent] if z_i == 1
                          else 1.0 - p_ind_given_f[z_parent])
        eta = b[expert_idx] + a * z_i
        logp_r = pt_ordinal_logp(ratings, kappa, eta)
        ll_zi.append(log_p_zi + pt.sum(logp_r))
    return pt.logaddexp(ll_zi[0], ll_zi[1])


def feature_log_marginal(
    ind_list: list[int],
    s_val: int,
    a: pt.TensorVariable,
    b: pt.TensorVariable,
    kappa: pt.TensorVariable,
    tree_ind_ratings: list[np.ndarray],
    tree_ind_experts: list[np.ndarray],
    p_feat_given_s: dict[int, float],
    p_ind_given_f: dict[int, float],
) -> pt.TensorVariable:
    """Log P(ratings_feature | stance=s_val, theta), marginalised over z_f."""
    ll_zf = []
    for z_f in [0, 1]:
        log_p_zf = np.log(p_feat_given_s[s_val] if z_f == 1
                          else 1.0 - p_feat_given_s[s_val])
        ll_inds = pt.zeros(())
        for ind_j in ind_list:
            r_j = pt.as_tensor_variable(tree_ind_ratings[ind_j])
            e_j = tree_ind_experts[ind_j]
            ll_inds = ll_inds + indicator_log_marginal(
                r_j, e_j, z_f, a, b, kappa, p_ind_given_f)
        ll_zf.append(log_p_zf + ll_inds)
    return pt.logaddexp(ll_zf[0], ll_zf[1])


def tree_log_marginal_lik(
    p_stance: pt.TensorVariable,
    a: pt.TensorVariable,
    b: pt.TensorVariable,
    kappa: pt.TensorVariable,
    feature_to_indicators: list[list[int]],
    tree_ind_ratings: list[np.ndarray],
    tree_ind_experts: list[np.ndarray],
    p_feat_given_s: dict[int, float],
    p_ind_given_f: dict[int, float],
) -> pt.TensorVariable:
    """Total log-marginal-likelihood for the mini DCM tree."""
    ll_s = []
    for s_val in [0, 1]:
        log_p_s = pt.log(p_stance) if s_val == 1 else pt.log(1.0 - p_stance)
        ll_features = pt.zeros(())
        for f_idx, ind_list in enumerate(feature_to_indicators):
            ll_features = ll_features + feature_log_marginal(
                ind_list, s_val, a, b, kappa,
                tree_ind_ratings, tree_ind_experts,
                p_feat_given_s, p_ind_given_f)
        ll_s.append(log_p_s + ll_features)
    return pt.logaddexp(ll_s[0], ll_s[1])


# Use it
with pm.Model() as mini_dcm_refactored:
    p_stance = pm.Beta("p_stance", alpha=1.0, beta=1.0)
    a = pm.HalfNormal("a", sigma=3.0)
    b_free = pm.Normal("b_free", mu=0.0, sigma=2.0, shape=n_tree_experts - 1)
    b = pt.concatenate([pt.zeros(1), b_free])
    kappa = pm.Normal("kappa", mu=0.0, sigma=2.0, shape=K - 1,
                      transform=pm.distributions.transforms.ordered,
                      initval=np.linspace(-1.5, 1.5, K - 1))

    ll = tree_log_marginal_lik(
        p_stance, a, b, kappa,
        feature_to_indicators, tree_ind_ratings, tree_ind_experts,
        p_feat_given_s, p_ind_given_f)
    pm.Potential("tree_ll", ll)

with mini_dcm_refactored:
    idata_refactored = pm.sample(draws=1000, tune=1500, chains=4,
                                  random_seed=42, target_accept=0.9)

print("p_stance posterior mean:",
      idata_refactored.posterior["p_stance"].mean().item():.3f)
```

</details>

---

## Summary

### What we built

| Component | Old approach | New approach |
|---|---|---|
| Expert input | Probability judgement (continuous) | 7-point ordinal rating |
| Per-indicator likelihood | Average probs → single Bernoulli | Ordinal probit per expert, shared cutpoints |
| Expert heterogeneity | Not modelled | Bias parameters $b_e$ |
| Latent state $z$ | Implicitly sampled via outer MC | Marginalised analytically via logsumexp |
| Inference | Multiple outer MC runs | Single MCMC run |

### Key implementation patterns

1. **`pt_ordinal_logp`**: computes ordered probit log-probabilities in pytensor using `pt.erfc`.
2. **`pm.Potential` + logsumexp**: marginalises discrete latent variables inside NUTS.
3. **Tree factorisation**: marginalise indicators within features, then features within stances. Each sum is over $\{0, 1\}$, so the cost is linear in the number of nodes, not exponential.
4. **Identifiability**: fix $b_1 = 0$, variance = 1, ordered cutpoints, $a > 0$.

### What comes next

- Scale to the full DCM tree (many stances, features, indicators).
- Learn the conditional probabilities $P(z_f \mid s)$ and $P(z_i \mid z_f)$ from data instead of fixing them.
- Add hierarchical priors on expert biases if the number of experts grows.
- Consider indicator-specific discrimination $a_j$ if different indicators have different diagnostic power.